# Interpretação e fairness

**Tarefas:** T3.5 · **Dono:** _(a definir)_

Bancada de trabalho: a lógica reutilizável fica em `src/`; aqui corre-se, explora-se e
guarda-se o que o `Projeto_Final.ipynb` vai ler (`reports/experiments.csv`, `reports/figures/`).
Regras da modelação: [docs/modelacao_base.md](../docs/modelacao_base.md).

Importância por permutação (`sklearn.inspection.permutation_importance`, PR-AUC) e métricas por idade, raça e sexo, tudo sobre as previsões OOF do treino. Sem explicação por doente (limitação a declarar no model card).

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # os caminhos de src/ (data/, reports/, models/) são relativos à raiz

import pandas as pd

from src.models.baselines import candidate_models
from src.models.evaluate import metrics_by_group, oof_permutation_importance
from src.pipeline import load_model_data

data = load_model_data()
best = candidate_models()["hist_gradient_boosting"]
oof = pd.read_csv(ROOT / "reports" / "oof_v1.csv")["oof_score"].to_numpy()  # gravado no 04

## Importância por permutação

Descida da PR-AUC na validação de cada fold quando a feature é baralhada (5 repetições); média e desvio entre os 5 folds.

In [2]:
imp = oof_permutation_importance(best, data)
imp.to_csv(ROOT / "reports" / "permutation_importance_v1.csv")
imp.head(15).round(4)

,importance,std
number_inpatient,0.0519,0.0078
discharge_disposition_id_desc,0.0496,0.0039
payer_code,0.0077,0.0011
medical_specialty,0.0056,0.0018
n_visits_prior,0.0052,0.0018
diag_1_group,0.0048,0.0015
admission_source_id_desc,0.0046,0.0016
diabetes_complicated,0.0026,0.0012
number_emergency,0.0025,0.0012
age_band,0.0024,0.0016


## Fairness por grupo

Top 10% calculado na população toda (OOF); por grupo vê-se a taxa de sinalizados e o recall. Grupos com menos de 100 encontros não se reportam.

In [3]:
{g: metrics_by_group(data.y_train, oof, data.X_train[g]).round(3)
 for g in ["age_band", "race", "gender"]}

{'age_band':            n  positive_rate  flagged_rate  recall  precision  pr_auc
 group                                                               
 0        129          0.023         0.008   0.000      0.000   0.055
 1        557          0.061         0.063   0.176      0.171   0.150
 2       1377          0.152         0.234   0.632      0.410   0.424
 3       3061          0.119         0.162   0.403      0.296   0.293
 4       7653          0.107         0.104   0.289      0.296   0.256
 5      13628          0.098         0.088   0.250      0.279   0.236
 6      17788          0.111         0.095   0.234      0.274   0.220
 7      20393          0.120         0.099   0.225      0.270   0.223
 8      13329          0.124         0.098   0.208      0.263   0.216
 9       2126          0.117         0.067   0.145      0.254   0.185,
 'race':                      n  positive_rate  flagged_rate  recall  precision  pr_auc
 group                                                     